<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-4-ICE-1/Unit_4_ICE_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Goal of This Notebook: Training Stability via Regularization

In this lesson, you'll explore two key techniques that help **neural networks train more reliably and generalize better**:

---

### L2 Regularization (also called weight decay)
This method discourages the model from assigning large weights by **adding a penalty term** to the loss function. It helps avoid overfitting.

> Think of it like applying gentle pressure to keep the model’s parameters close to zero — nudging it toward simpler, more generalizable solutions.

---

### Gradient Clipping
When training deep or wide networks, the gradients can sometimes become very large (explode), leading to unstable or diverging learning. **Gradient clipping** rescales them to stay within a safe range.

> It’s like placing a “speed limit” on how fast your model is allowed to learn in any single update.

---

### What You'll Do in This Notebook:
1. Train a simple model **with L2 regularization** and visualize how its weights behave.
2. Train a large model **with gradient clipping**, and inspect the total gradient norm before and after clipping.


# Training with Regularization in PyTorch
This notebook demonstrates two regularization techniques:
- **L2 Regularization** using `weight_decay`
- **Gradient Clipping** using `clip_grad_norm_`

We'll walk through both using small synthetic models.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

## L2 Regularization Example

In [ ]:
# Define a simple linear model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        return self.fc(x)

net = Net()

In [ ]:
# Create synthetic data
x = torch.unsqueeze(torch.linspace(-1, 1, 100), dim=1)
y = 3 * x + 0.1 * torch.randn(x.size())

In [ ]:
# Define optimizer with L2 regularization (weight decay)
optimizer = optim.SGD(net.parameters(), lr=0.01, weight_decay=0.005)
loss_fn = nn.MSELoss()

In [ ]:
# Training loop with L2 penalty
for epoch in range(100):
    optimizer.zero_grad()
    y_pred = net(x)
    loss = loss_fn(y_pred, y)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [ ]:

# 📈 Visualizing weight growth with and without L2 regularization

# Extract learned weights from the model
with torch.no_grad():
    learned_weight = net.fc.weight.item()
    learned_bias = net.fc.bias.item()

print(f"Learned weight: {learned_weight:.4f}")
print(f"Learned bias: {learned_bias:.4f}")

# Plot prediction vs. original data
plt.figure(figsize=(6,4))
plt.scatter(x.numpy(), y.numpy(), label='Data')
plt.plot(x.numpy(), net(x).detach().numpy(), color='red', label='Model Prediction')
plt.legend()
plt.title('Effect of L2 Regularization on Linear Fit')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.show()



### What to Notice:
- The red line represents the model's prediction after training.
- Even with noise in the data, the model avoids overfitting by keeping weights small.
- If you remove the `weight_decay` from the optimizer, the model might overfit or learn unstable weights.


## Gradient Clipping Example

In [ ]:
# Define large synthetic model
batch_size, dim_in, dim_h, dim_out = 128, 2000, 200, 20
input_X = torch.randn(batch_size, dim_in)
output_Y = torch.randn(batch_size, dim_out)

model = nn.Sequential(
    nn.Linear(dim_in, dim_h),
    nn.ReLU(),
    nn.Linear(dim_h, dim_out),
)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss(reduction='sum')

In [ ]:
# One training step with gradient clipping
optimizer.zero_grad()
pred_y = model(input_X)
loss = loss_fn(pred_y, output_Y)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0, norm_type=2.0)
optimizer.step()

In [ ]:
# Check total gradient norm
total_norm = 0.0
for p in model.parameters():
    param_norm = p.grad.detach().data.norm(2)
    total_norm += param_norm.item() ** 2
total_norm = total_norm ** 0.5
print(f"Total gradient norm after clipping: {total_norm:.4f}")

In [ ]:

# Visualizing the effect of gradient clipping

# Measure gradient norm before and after clipping again for visibility
optimizer.zero_grad()
pred_y = model(input_X)
loss = loss_fn(pred_y, output_Y)
loss.backward()

# Compute total norm BEFORE clipping
pre_clip_norm = 0.0
for p in model.parameters():
    if p.grad is not None:
        param_norm = p.grad.detach().data.norm(2)
        pre_clip_norm += param_norm.item() ** 2
pre_clip_norm = pre_clip_norm ** 0.5

# Apply clipping
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0, norm_type=2.0)

# Compute total norm AFTER clipping
post_clip_norm = 0.0
for p in model.parameters():
    if p.grad is not None:
        param_norm = p.grad.detach().data.norm(2)
        post_clip_norm += param_norm.item() ** 2
post_clip_norm = post_clip_norm ** 0.5

print(f"Gradient norm before clipping: {pre_clip_norm:.4f}")
print(f"Gradient norm after clipping:  {post_clip_norm:.4f}")



### What to Notice:
- Gradient norms are often **very large** in high-dimensional models, especially before the model stabilizes.
- Clipping helps keep updates within a safe range.
- Without clipping, these gradients could lead to **unstable parameter updates** or **loss spikes**.

> You can try setting `max_norm=1.0` or removing the clip call to compare the impact!



## Interactive Experiment: L2 Regularization Tuner

Use the sliders below to change the **learning rate** and **L2 penalty (weight decay)**.  
You'll see how these affect the model’s predictions and stability.


In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

def train_l2_demo(learning_rate=0.01, weight_decay=0.01):
    model = Net()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    for epoch in range(100):
        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        pred = model(x).detach().numpy()
        learned_w = model.fc.weight.item()
        learned_b = model.fc.bias.item()

    plt.figure(figsize=(6, 4))
    plt.scatter(x.numpy(), y.numpy(), label='Data')
    plt.plot(x.numpy(), pred, 'r-', label='Prediction')
    plt.title(f"LR={learning_rate}, L2={weight_decay}, w={learned_w:.2f}, b={learned_b:.2f}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.grid(True)
    plt.show()

widgets.interact(train_l2_demo,
                 learning_rate=widgets.FloatSlider(min=0.001, max=0.1, step=0.005, value=0.01),
                 weight_decay=widgets.FloatSlider(min=0.0, max=0.1, step=0.005, value=0.01));



## Interactive Experiment: Gradient Clipping Tuner

Try adjusting the **gradient norm cap** and **learning rate** to see how they influence training behavior and stability.


In [ ]:

def train_clip_demo(learning_rate=1e-4, clip_norm=5.0):
    model = nn.Sequential(
        nn.Linear(dim_in, dim_h),
        nn.ReLU(),
        nn.Linear(dim_h, dim_out),
    )
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss(reduction='sum')

    loss_history = []
    grad_norm_history = []

    for epoch in range(30):  # Shorter loop for visualization
        optimizer.zero_grad()
        pred_y = model(input_X)
        loss = loss_fn(pred_y, output_Y)
        loss.backward()

        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        grad_norm_history.append(total_norm)

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm, norm_type=2.0)
        optimizer.step()

        loss_history.append(loss.item())

    # Plot loss and gradient norm
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(loss_history, label='Training Loss')
    ax1.set_title("Loss over Epochs")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True)

    ax2.plot(grad_norm_history, label='Gradient Norm (Pre-Clipping)', color='orange')
    ax2.axhline(y=clip_norm, color='red', linestyle='--', label='Clip Threshold')
    ax2.set_title("Gradient Norm Before Clipping")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("L2 Norm")
    ax2.legend()
    ax2.grid(True)

    plt.suptitle(f"Clip Norm: {clip_norm}, LR: {learning_rate}")
    plt.tight_layout()
    plt.show()

widgets.interact(train_clip_demo,
                 learning_rate=widgets.FloatSlider(min=1e-5, max=1e-3, step=1e-5, value=1e-4),
                 clip_norm=widgets.FloatSlider(min=1.0, max=10.0, step=0.5, value=5.0));



## Gradient Clipping: Side-by-Side Comparison Mode

This interactive demo runs two identical models side-by-side:
- **Left**: Model without gradient clipping
- **Right**: Model with clipping applied

You'll compare:
- How loss curves behave over time
- Whether gradient norms exceed safe thresholds

Try increasing the learning rate or lowering the clip threshold to exaggerate the difference.


In [ ]:
def compare_clipping(learning_rate=1e-4, clip_norm=5.0):
    def build_model():
        return nn.Sequential(
            nn.Linear(dim_in, dim_h),
            nn.ReLU(),
            nn.Linear(dim_h, dim_out),
        )

    model_unclipped = build_model()
    model_clipped = build_model()

    opt_unclipped = optim.Adam(model_unclipped.parameters(), lr=learning_rate)
    opt_clipped = optim.Adam(model_clipped.parameters(), lr=learning_rate)

    loss_fn = nn.MSELoss(reduction='sum')

    loss_unclipped, grad_norm_unclipped = [], []
    loss_clipped, grad_norm_clipped_pre, grad_norm_clipped_post = [], [], []

    for epoch in range(30):
        # --- Unclipped ---
        opt_unclipped.zero_grad()
        out_uc = model_unclipped(input_X)
        loss_uc = loss_fn(out_uc, output_Y)
        loss_uc.backward()
        norm_uc = sum(p.grad.detach().norm(2).item()**2 for p in model_unclipped.parameters() if p.grad is not None) ** 0.5
        opt_unclipped.step()

        # --- Clipped ---
        opt_clipped.zero_grad()
        out_c = model_clipped(input_X)
        loss_c = loss_fn(out_c, output_Y)
        loss_c.backward()
        norm_pre = sum(p.grad.detach().norm(2).item()**2 for p in model_clipped.parameters() if p.grad is not None) ** 0.5
        torch.nn.utils.clip_grad_norm_(model_clipped.parameters(), max_norm=clip_norm)
        norm_post = sum(p.grad.detach().norm(2).item()**2 for p in model_clipped.parameters() if p.grad is not None) ** 0.5
        opt_clipped.step()

        # Log data
        loss_unclipped.append(loss_uc.item())
        loss_clipped.append(loss_c.item())
        grad_norm_unclipped.append(norm_uc)
        grad_norm_clipped_pre.append(norm_pre)
        grad_norm_clipped_post.append(norm_post)

    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # --- Loss plot ---
    axs[0].plot(loss_unclipped, '--', label='No Clipping', color='blue')
    axs[0].plot(loss_clipped, '-', label='With Clipping', color='green')
    axs[0].set_title("Training Loss")
    axs[0].set_xlabel("Epoch")
    axs[0].set_ylabel("Loss")
    axs[0].legend()
    axs[0].grid(True)

    # --- Gradient norms ---
    axs[1].plot(grad_norm_clipped_pre, color='orange', label='Pre-Clipping')
    axs[1].plot(grad_norm_clipped_post, color='green', label='Post-Clipping')
    axs[1].plot(grad_norm_unclipped, '--', color='blue', label='No Clipping')
    axs[1].axhline(y=clip_norm, linestyle='--', color='red', label='Clip Threshold')
    axs[1].set_title("Gradient Norms")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("L2 Norm")
    axs[1].legend()
    axs[1].grid(True)

    plt.suptitle(f"Clip Norm = {clip_norm}, Learning Rate = {learning_rate}")
    plt.tight_layout()
    plt.show()

widgets.interact(compare_clipping,
                 learning_rate=widgets.FloatSlider(min=1e-5, max=5e-4, step=1e-5, value=1e-4),
                 clip_norm=widgets.FloatSlider(min=1.0, max=10.0, step=0.5, value=5.0));
